# 01 — Preprocess TACO + Roboflow

Notebook này tạo dữ liệu hoàn chỉnh cho train: COCO 7-class, YOLO 7-class, YOLO binary waste, crop dataset 7-class. Roboflow được xử lý riêng và chỉ bật khi bạn đã cấu hình đúng path/config.


# Common setup notes

Trước khi chạy notebook:

1. Bật GPU trong Kaggle: `Settings → Accelerator → GPU`.
2. Thêm dataset TACO/Roboflow ở mục `Add Input`.
3. Nếu dùng W&B online, thêm Kaggle Secret:
   - Name: `WANDB_API_KEY`
   - Value: API key của tài khoản W&B.
4. Sửa biến `REPO_URL`, `WANDB_ENTITY`, `TACO_ROOT` hoặc `ROBOFLOW_ROOT` cho đúng với nhóm bạn.

Các notebook này dùng nguyên command-line scripts trong GitHub repo đã hoàn thiện, không viết lại training logic trong notebook.


In [ ]:
import os
from pathlib import Path

# ===== EDIT ME =====
REPO_URL = "https://github.com/nnnhuan251-hcmus/Detect-Waste-ADN.git"
REPO_DIR = Path("/kaggle/working/Detect-Waste-ADN")
BRANCH = "main"

# TACO config
DATA_CONFIG = "configs/data/taco_7class.yaml"

# Optional Roboflow
ROBOFLOW_ENABLED = True
ROBOFLOW_ROOT = "/kaggle/input/<roboflow-dataset-folder>"
ROBOFLOW_CONFIG = "configs/data/taco_7class.yaml"

os.environ["REPO_URL"] = REPO_URL
os.environ["REPO_DIR"] = str(REPO_DIR)
os.environ["BRANCH"] = BRANCH
os.environ["DATA_CONFIG"] = DATA_CONFIG
os.environ["ROBOFLOW_ENABLED"] = "1" if ROBOFLOW_ENABLED else "0"
os.environ["ROBOFLOW_ROOT"] = ROBOFLOW_ROOT
os.environ["ROBOFLOW_CONFIG"] = ROBOFLOW_CONFIG

print("REPO_DIR:", REPO_DIR)
print("DATA_CONFIG:", DATA_CONFIG)
print("ROBOFLOW_ENABLED:", ROBOFLOW_ENABLED)
print("ROBOFLOW_ROOT:", ROBOFLOW_ROOT)


In [ ]:
%%bash
set -e

cd /kaggle/working

if [ ! -d "$REPO_DIR/.git" ]; then
  echo "Cloning repo..."
  git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"
else
  echo "Repo exists. Pulling latest changes..."
  cd "$REPO_DIR"
  git fetch origin "$BRANCH"
  git checkout "$BRANCH"
  git pull origin "$BRANCH"
fi

cd "$REPO_DIR"

python -m pip install -q --upgrade pip
pip install -q -r requirements.txt

if [ -f requirements-optional.txt ]; then
  pip install -q -r requirements-optional.txt || true
fi

export PYTHONPATH="$REPO_DIR/src:$PYTHONPATH"
echo "PYTHONPATH=$PYTHONPATH"


In [ ]:
%%bash
set -e

cd "$REPO_DIR"
export PYTHONPATH="$REPO_DIR/src:$PYTHONPATH"

echo "=== Compile check ==="
python -m compileall -q src scripts

echo "=== Core import check ==="
python - <<'PY'
import torch
from pathlib import Path

from waste_detection.config.config_loader import ConfigLoader
from waste_detection.data.coco_reader import CocoReader
from waste_detection.data.coco_validator import COCOValidator
from waste_detection.data.crop_dataset import CropClassificationDataset
from waste_detection.models.efficientnet_classifier import EfficientNetB0Classifier
from waste_detection.training.detector_trainer import DetectorTrainer
from waste_detection.training.classifier_trainer import ClassifierTrainer
from waste_detection.inference.hybrid_predictor import HybridPredictor
from waste_detection.evaluation.classification_metrics import ClassificationMetrics
from waste_detection.evaluation.hybrid_metrics import HybridMetrics

print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
print("core imports passed")
PY


## 1. Kiểm tra input dataset trên Kaggle

Cell này không sửa gì, chỉ giúp xác định path thật của TACO/Roboflow.


In [ ]:
import os
import glob
import kagglehub

repo_dir = os.environ.get("REPO_DIR", "/kaggle/working/Detect-Waste-ADN")
raw_dir = os.path.join(repo_dir, "data", "raw")
os.makedirs(raw_dir, exist_ok=True)
target_taco = os.path.join(raw_dir, "TACO")

# 1. Thử tìm TACO dataset trong /kaggle/input (nếu bạn Add Input thủ công)
input_taco = None
for path in glob.glob("/kaggle/input/**/annotations.json", recursive=True):
    input_taco = os.path.dirname(path)
    break

if input_taco:
    print(f"✅ Tìm thấy TACO dataset qua Add Input tại: {input_taco}")
    taco_path = input_taco
else:
    print("⏳ Không thấy dữ liệu Add Input. Đang tự động tải TACO dataset từ Kaggle...")
    taco_path = kagglehub.dataset_download("wilmacv251/taco-dataset-waste")
    print("✅ Tải tự động hoàn tất tại:", taco_path)

if os.path.exists(target_taco):
    os.system(f"rm -rf {target_taco}")
os.system(f"ln -s {taco_path} {target_taco}")
print("👉 Đã link data thành công vào:", target_taco)


## 2. Load config và kiểm tra path

Nếu lỗi ở đây, sửa `configs/data/taco_7class_kaggle.yaml` trước khi chạy preprocessing.


In [ ]:
%%bash
set -e

cd "$REPO_DIR"
export PYTHONPATH="$REPO_DIR/src:$PYTHONPATH"

python - <<'PY'
from pathlib import Path
import os
from waste_detection.config.config_loader import ConfigLoader

data_config_path = Path(os.environ["DATA_CONFIG"])
loader = ConfigLoader(project_root="/kaggle/working/Detect-Waste-ADN")
cfg = loader.load_all(
    data_config_path=str(data_config_path),
    log_file_name="preprocess_config_check.log",
)
dc = cfg.data
print("annotation_file:", dc.paths.annotation_file, "| exists:", dc.paths.annotation_file.exists())
print("image_root:", dc.paths.image_root, "| exists:", dc.paths.image_root.exists())
print("processed_root:", dc.paths.processed_root)
if not dc.paths.annotation_file.exists():
    raise SystemExit("Không tìm thấy annotation_file. Sửa data config.")
if not dc.paths.image_root.exists():
    raise SystemExit("Không tìm thấy image_root. Sửa data config.")
print("TACO data config passed.")
PY


## 3. Chạy TACO preprocessing end-to-end

Có `set -e`: nếu `check_coco.py` lỗi thì notebook dừng, không chạy tiếp `prepare_taco.py`.


In [ ]:
%%bash
set -e

cd "$REPO_DIR"
export PYTHONPATH="$REPO_DIR/src:$PYTHONPATH"

echo "=== Check COCO raw ==="
python scripts/data/check_coco.py \
  --data-config "$DATA_CONFIG"

echo
echo "=== Clean old processed outputs ==="
rm -rf data/processed outputs/logs/prepare_taco.log

echo
echo "=== Prepare TACO end-to-end ==="
python scripts/data/prepare_taco.py \
  --data-config "$DATA_CONFIG"


## 4. Verify TACO processed outputs


In [ ]:
%%bash
set -e

cd "$REPO_DIR"

echo "=== Processed folders ==="
ls -lah data/processed || true
find data/processed -maxdepth 3 -type f \( -name "*.json" -o -name "data.yaml" -o -name "*.csv" \) | sort | head -200

echo
echo "=== YOLO data.yaml ==="
cat data/processed/yolo_7class/data.yaml
echo
cat data/processed/yolo_binary_waste/data.yaml

echo
echo "=== Crop counts ==="
python - <<'PY'
from pathlib import Path
classes = ["plastic", "paper", "metal", "glass", "organic", "cigarette", "other"]
root = Path("data/processed/crops_7class")
for split in ["train", "val", "test"]:
    print(f"\n[{split}]")
    for cls in classes:
        n = len([p for p in (root/split/cls).rglob("*") if p.is_file()])
        print(f"{cls:10s}: {n}")
PY


## 5. Optional — Roboflow preprocessing

Mặc định cell này **skip** để tránh lỗi path. Chỉ bật `ROBOFLOW_ENABLED=True` ở cell đầu khi bạn đã có `ROBOFLOW_ROOT` và `configs/data/roboflow_7class_kaggle.yaml` đúng.


In [ ]:
import os

roboflow_root = os.environ.get("ROBOFLOW_ROOT", "")

# Kiểm tra xem đường dẫn có thật không và không phải là template ảo
if os.path.exists(roboflow_root) and "<roboflow" not in roboflow_root:
    print(f"✅ Tìm thấy Roboflow dataset qua Add Input tại: {roboflow_root}")
else:
    print("⏳ Không thấy dữ liệu thủ công. Đang tự động tải từ Roboflow API...")
    os.system("pip install -q roboflow")
    from roboflow import Roboflow
    
    rf = Roboflow(api_key="5r1QbIwiFST8QcGoDpot")
    project = rf.workspace("rf100-vl").project("taco-trash-annotations-in-context-dtyly-awiq")
    version = project.version(2)
    dataset = version.download("coco", location="/kaggle/working/roboflow_data")
    
    os.environ["ROBOFLOW_ROOT"] = dataset.location
    print("✅ Tải tự động hoàn tất tại:", dataset.location)


In [ ]:
%%bash
set -e

cd "$REPO_DIR"
export PYTHONPATH="$REPO_DIR/src:$PYTHONPATH"

if [ "$ROBOFLOW_ENABLED" != "1" ]; then
  echo "ROBOFLOW_ENABLED=0 → Skip Roboflow preprocessing."
  exit 0
fi

test -d "$ROBOFLOW_ROOT" || (echo "ROBOFLOW_ROOT không tồn tại: $ROBOFLOW_ROOT" && exit 1)
test -f "$ROBOFLOW_CONFIG" || (echo "ROBOFLOW_CONFIG không tồn tại: $ROBOFLOW_CONFIG" && exit 1)

# Tự động thêm category ảo của Roboflow vào danh sách ignore trong mapping_label.json
python - <<'PY'
import json
map_file = "configs/data/mapping_label.json"
with open(map_file, "r", encoding="utf-8") as f:
    data = json.load(f)
if "taco-trash-annotations-in-context-dtyly-awiq" not in data["ignore"]:
    data["ignore"].append("taco-trash-annotations-in-context-dtyly-awiq")
with open(map_file, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)
PY

echo "=== Import Roboflow COCO ==="
python scripts/data/import_roboflow_coco.py \
  --data-config "$ROBOFLOW_CONFIG" \
  --dataset-dir "$ROBOFLOW_ROOT" \
  --mapping-label configs/data/mapping_label.json \
  --output-coco-dir data/interim/external/roboflow_coco_7class \
  --output-image-dir data/interim/external/roboflow_images

echo "=== Convert Roboflow COCO to YOLO ==="
python scripts/data/convert_coco_to_yolo.py \
  --data-config "$ROBOFLOW_CONFIG"

echo "=== Create Roboflow crop dataset ==="
python scripts/data/create_crop_dataset.py \
  --data-config "$ROBOFLOW_CONFIG"


## 6. Khuyến nghị gộp TACO + Roboflow

Không gộp mặc định trong notebook này. Khuyến nghị an toàn cho báo cáo:

- Main experiment: train/evaluate chuẩn trên TACO.
- Extended experiment: chỉ gộp **train split** TACO + Roboflow, giữ TACO test làm test chính.
- Không gộp toàn bộ rồi split lại nếu chưa kiểm tra ảnh trùng, vì dễ data leakage.


In [ ]:
from pathlib import Path
import json, os

summary = {
    "recommendation": "Use TACO as main benchmark. Use Roboflow/combined training as extended experiment.",
    "safe_combination_rule": "Combine training splits only; keep TACO validation/test unchanged for fair comparison.",
    "saved_outputs": [
        "data/processed/coco_7class",
        "data/processed/yolo_7class",
        "data/processed/yolo_binary_waste",
        "data/processed/crops_7class",
        "outputs/logs",
        "outputs/metrics/data"
    ]
}
Path("outputs/metrics").mkdir(parents=True, exist_ok=True)
Path("outputs/metrics/preprocessing_plan.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8"
)
summary


## 7. Đóng gói output cần chia cho notebook train


In [ ]:
%%bash
set -e

cd "$REPO_DIR"

mkdir -p /kaggle/working/artifacts

tar -czf /kaggle/working/artifacts/preprocessed_data_and_reports.tar.gz \
  data/processed \
  outputs/logs \
  outputs/metrics \
  2>/dev/null || true

ls -lh /kaggle/working/artifacts
